Transformation → Silver
Source → Bronze (bronze.crm_customers)
Target → silver.crm_customers

1. Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

2. Read Bronze Tables

In [0]:
df_bronze = spark.read.table("bronze.crm_customers")

In [0]:
display(df_bronze
    )

2. Flatten JSON

In [0]:
df_flat = df_bronze.select(
    # Primary key
    F.col("customer_id"),

    # Personal info
    F.col("personal_info.first_name").alias("first_name"),
    F.col("personal_info.last_name").alias("last_name"),
       F.concat_ws(" ",
        F.col("personal_info.first_name"),
        F.col("personal_info.last_name")
    ).alias("customer_name"),
    F.col("contact.addresses")[0]["city"].alias("city"),
    F.col("contact.addresses")[0]["country"].alias("country"),
    F.col("personal_info.gender").alias("gender"),
    F.col("personal_info.date_of_birth").alias("date_of_birth"),

    # Contact
    F.col("contact.email").alias("email"),
    F.col("contact.phone").alias("phone"),

    # Loyalty
    F.col("loyalty.tier").alias("loyalty_tier"),
    F.col("loyalty.points").alias("loyalty_points"),
    F.col("loyalty.join_date").alias("loyalty_join_date"),

    # Preferences
    F.col("preferences.preferred_channel").alias("preferred_channel"),

    # Metadata
    F.col("metadata.created_at").alias("created_at"),


    F.col("ingestion_timestamp"),
    F.col("source_file")
)

In [0]:
display(df_flat)

3. Data Type Casting

In [0]:
df_typed = df_flat \
    .withColumn("customer_id", F.col("customer_id").cast("string")) \
    .withColumn("customer_name", F.col("customer_name").cast("string")) \
    .withColumn("email", F.lower(F.col("email"))) \
    .withColumn("phone", F.col("phone").cast("string")) \
    .withColumn("gender", F.col("gender").cast("string")) \
    .withColumn("date_of_birth", F.to_date("date_of_birth")) \
    .withColumn("loyalty_points", F.col("loyalty_points").cast("int")) \
    .withColumn("loyalty_tier", F.col("loyalty_tier").cast("string")) \
    .withColumn("loyalty_join_date", F.to_date("loyalty_join_date")) \
    .withColumn("preferred_channel", F.col("preferred_channel").cast("string")) \
    .withColumn("created_at", F.to_timestamp("created_at")) \
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp"))

In [0]:

# Key Standardization (customer_id)

from pyspark.sql.functions import regexp_replace, col

df_typed = df_typed.withColumn(
    "customer_id",
    regexp_replace(
        regexp_replace(col("customer_id"), "^CUST_", ""),
        "^0+",
        ""
    )
)

4. Handle Null Values

In [0]:
df_clean = df_typed.dropna(subset=["customer_id"])

5. Deduplication

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy(F.col("ingestion_timestamp").desc())

df_dedup = df_clean \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num")

6. Data Quality Rules

In [0]:
df_valid = df_dedup.filter(
    F.col("customer_id").isNotNull()
)

7. Write to Silver 

In [0]:
spark.sql("DROP TABLE IF EXISTS silver.crm_customers")

df_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.crm_customers")

8. Validation

In [0]:
spark.read.table("silver.crm_customers").count()
display(spark.read.table("silver.crm_customers"))

In [0]:
spark.read.table("silver.crm_customers") \
    .groupBy("customer_id") \
    .count() \
    .filter("count > 1") \
    .show()
    

In [0]:
spark.read.table("silver.crm_customers").count()

In [0]:
spark.table("silver.crm_customers").select("customer_id").show(5)